# 03 — G0 pseudo-glosses and TRAIN-only vocabularies

**Purpose.** Write G0 v0.1 drafts for all 400 sentences (status `auto_draft`). These are **pseudo-glosses made by a rule from the Bangla text — NOT BdSL annotation**, nobody checked them against the videos. Build the vocabularies from TRAIN only.

**Kaggle settings.** Accelerator **None** · Internet Off is fine.

**Inputs.** Dataset `bdsl-k400` + **Add Input → Output of notebook 02** (carries the features forward).

**Outputs** (under `/kaggle/working/k400/`, saved by *Save Version → Save & Run All*). `processed/glosses/gloss_g0_{train,val,test}.csv`, `processed/vocab/*.json`, `vocab_report.json`.

**Expected run time** [ESTIMATE, not measured yet]. under 2 minutes.

**What to check.** `compare_with_committed`: IDENTICAL for train, val and test (the same drafts as the desktop made).

> Honesty rules: label every number measured or estimated; results are small-scale and signer-dependent;
> the test split is used once (notebook 07); low coordinate error is not proof of correct sign language;
> G0 glosses are pseudo-glosses, not BdSL annotation; no human or expert validation exists.

In [ ]:
# Cell 1 - Setup (the same in every notebook). Finds the data and the code, sets the paths, and copies the
# outputs of earlier notebooks (added with "Add Input") into /kaggle/working/k400 so results accumulate.
import os, sys, zipfile, pathlib, json
NOTEBOOK = "03_gloss_g0"

def _find_root():
    base = pathlib.Path("/kaggle/input")
    if base.is_dir():
        for dirpath, dirnames, filenames in os.walk(base):
            if "UPLOAD_MANIFEST.csv" in filenames and "PACKAGE_INFO.json" in filenames:
                return pathlib.Path(dirpath)
            depth = len(pathlib.Path(dirpath).relative_to(base).parts)
            dirnames[:] = [] if depth >= 4 else [d for d in dirnames if d not in ("videos", "k400")]
        zips = sorted(base.rglob("bdsl-k400-*.zip"))
        if zips:  # Kaggle kept the upload as one zip file: unpack it once (a few minutes)
            dest = pathlib.Path("/tmp/k400_dataset")   # /tmp is not saved as Output
            if not (dest / "UPLOAD_MANIFEST.csv").is_file():
                print("Unpacking", zips[0], "->", dest)
                zipfile.ZipFile(zips[0]).extractall(dest)
            return dest
        raise FileNotFoundError("The private dataset is not attached. Right panel: Add Input -> Your Datasets -> bdsl-k400.")
    here = pathlib.Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / "pyproject.toml").is_file() and (c / "kaggle" / "src" / "k400").is_dir():
            return c
    raise FileNotFoundError("Open this notebook inside the project folder, or on Kaggle with the dataset attached.")

DATASET_ROOT = _find_root()
CODE_ROOT = DATASET_ROOT / "code" if (DATASET_ROOT / "code" / "pyproject.toml").is_file() else DATASET_ROOT
for _p in (CODE_ROOT / "kaggle" / "src", CODE_ROOT / "src"):
    sys.path.insert(0, str(_p))
from k400.setup import setup
S = setup(DATASET_ROOT, CODE_ROOT, NOTEBOOK)

In [ ]:
rc_gloss = S.k400("gloss", "--compare-with", str(CODE_ROOT / "kaggle/processed/glosses"))

In [ ]:
import pandas as pd
from IPython.display import display
g = pd.read_csv(S.state_root / "processed/glosses/gloss_g0_train.csv", encoding="utf-8-sig")
print("LABEL: G0 pseudo-gloss (auto_draft) - NOT a BdSL annotation")
display(g[["index", "sentence_norm", "gloss_auto", "tags_auto", "status"]].head(10))
print(json.dumps(json.load(open(S.state_root / "processed/vocab/vocab_report.json")), indent=1, ensure_ascii=False))

In [ ]:
S.finish(checks=[
    ("G0 files identical to the committed copies", rc_gloss == 0),
    ("320 + 40 + 40 rows, all auto_draft", all(len(pd.read_csv(S.state_root / f"processed/glosses/gloss_g0_{s}.csv")) == n for s, n in (("train", 320), ("val", 40), ("test", 40)))),
])